<a href="https://colab.research.google.com/github/EhlertMoracha/SemProject_MLEngine/blob/main/support_needs_ml_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI-Powered Chronic Disease Support Needs Identification: ML Engine

**Goal.** Read a patient's demographic, health, and social answers and estimate which of five **non-medical support needs** they may have: emotional, social, food, financial, and transportation.

**Output.** For every patient the engine returns a probability for each need, a yes/no flag per need, and the main answers that pushed each flagged need up.

**Data.** CDC National Health Interview Survey (NHIS) 2023, Sample Adult file.

**How to run (Google Colab).** Run the cells from top to bottom. When asked, upload `adult23.csv` (unzipped from `adult23csv.zip` on the CDC NHIS website). Expect roughly 15 to 25 minutes in total, mostly the tuning and SHAP cells.

**Scope.** The engine suggests *possible* needs for a person to review. It does not diagnose, prescribe, or replace a healthcare professional.

## 1. Approach and theory (in plain words)

**Multi-label classification.** Normal classification picks one answer ("cat or dog"). Here one patient can need several kinds of help at once, so we predict five yes/no answers together.

**Binary Relevance.** We train one separate yes/no model for each of the five needs. It is simple, gives one probability per need, and lets the patient accept or decline each suggestion on its own.

**Random Forest (base model).** A decision tree asks a chain of questions ("is income low? is health poor?") and ends in an answer. A Random Forest grows many different trees and lets them vote. It copes well with survey data that mixes numbers and categories, picks up non-linear patterns, and overfits less than a single tree.

**Comparison models.** *Decision Tree* (simple baseline), *Logistic Regression* (a straight-line style model), and a *"predict no needs for everyone"* baseline that any useful model must beat. We also test two other multi-label strategies (native multi-output forest and Classifier Chains).

**Validation.**
- **Hold-out test set (20%)**: patients the model never saw, used once for the final score.
- **5-fold cross-validation**: the training data is split five ways, and the model is trained and scored five times, so the result does not depend on one lucky split.
- **Per-need thresholds**: the model outputs a probability, and we choose the cut-off for "flag this need" separately for each need.

**Metrics.** *Precision* = of the needs we flag, how many are real. *Recall* = of the real needs, how many we catch. *F1* combines both. *Macro F1* averages F1 over the five needs equally (so rare needs count fully), and *Micro F1* pools all decisions. *Hamming loss* is the share of wrong yes/no decisions, and *subset accuracy* needs all five right at once. Macro F1 is our main metric because missing someone who needs help is the costly mistake.

In [ ]:
# WHAT: Install the SHAP library, which explains the model's predictions (Colab does not include it by default).
# HOW/WHY: Run this once per session; the -q flag keeps the output short.
!pip install -q shap

In [ ]:
# WHAT: Import every library used in this notebook: data handling, plotting, scikit-learn models and metrics.
# HOW/WHY: Keeping all imports in one cell lets the later cells focus on the steps instead of set-up.
import os, json, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, shap
import sklearn
from scipy.stats import randint
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, f1_score,
                             hamming_loss, precision_score, recall_score)
from sklearn.model_selection import KFold, RandomizedSearchCV, cross_validate, train_test_split
from sklearn.multioutput import ClassifierChain, MultiOutputClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
print("scikit-learn version:", sklearn.__version__)

## 2. Data

We use the **NHIS 2023 Sample Adult** file (about 29,500 adults, 647 variables). It was chosen over BRFSS 2022 because it asks about all five support areas fairly directly, covers chronic conditions well, and is a national sample. Each variable's meaning and codes come from the NHIS *codebook* and *variable summary* (same year).

In [ ]:
# WHAT: Load the NHIS 2023 Sample Adult CSV into a table (DataFrame); in Colab, ask for the file if it is not there yet.
# HOW/WHY: low_memory=False reads the whole wide file at once, which avoids mixed-type warnings across its 647 columns.
DATA_PATH = "adult23.csv"
if not os.path.exists(DATA_PATH):
    from google.colab import files
    files.upload()          # choose adult23.csv when the file picker opens

df = pd.read_csv(DATA_PATH, low_memory=False)
print(df.shape)             # expect about (29522, 647)

## 3. Choosing the variables

NHIS has no ready-made "needs support" column, so we **build five labels from survey questions**. Each label is 1 if the patient answers *any* of its questions in the "need" direction.

| Label | Source questions | Rule for "need" = 1 |
|---|---|---|
| Emotional | `ANXFREQ_A`, `DEPFREQ_A`, `PHQ2SCREEN_A`, `GAD2SCREEN_A` | Anxious or depressed daily/weekly, or a positive PHQ-2 / GAD-2 screen |
| Social (proxy) | `SOCSCLPAR_A`, `SOCERRNDS_A` | "A lot" of difficulty joining social activities or doing errands alone |
| Food | `FDSCAT3_A` | Low or very low food security |
| Financial | `PAYBLL12M_A`, `MEDDL12M_A`, `MEDNG12M_A`, `RXDG12M_A`, `HOUSECOST_A` | Trouble paying medical bills, care or medication skipped because of cost, trouble with housing costs |
| Transportation | `TRANSPOR_A` | Delayed medical care because of no transportation |

**Important limitation.** The 2023 file has no loneliness or social-support question, so the Social label measures *difficulty with social participation and daily tasks*, not isolation itself.

**Chronic-condition filter.** We keep only adults who report at least one of: hypertension, coronary heart disease, angina, heart attack, stroke, current asthma, cancer, diabetes, COPD, arthritis, or high cholesterol.

In [ ]:
# WHAT: List the variables used for labels and the chronic-condition filter, and write each label rule as data.
# HOW/WHY: Keeping the rules in a dictionary {variable: codes that signal a need} makes them easy to audit against the codebook and change later.
label_sources = ["ANXFREQ_A", "DEPFREQ_A", "PHQ2SCREEN_A", "GAD2SCREEN_A",
                 "SOCSCLPAR_A", "SOCERRNDS_A", "FDSCAT3_A",
                 "PAYBLL12M_A", "MEDDL12M_A", "MEDNG12M_A", "RXDG12M_A",
                 "HOUSECOST_A", "TRANSPOR_A"]

chronic_vars = ["HYPEV_A", "CHDEV_A", "ANGEV_A", "MIEV_A", "STREV_A", "ASTILL_A",
                "CANEV_A", "DIBEV_A", "COPDEV_A", "ARTHEV_A", "CHLEV_A"]

rules = {
    "emotional": {"ANXFREQ_A": [1, 2], "DEPFREQ_A": [1, 2], "PHQ2SCREEN_A": [1], "GAD2SCREEN_A": [1]},
    "social":    {"SOCSCLPAR_A": [3, 4], "SOCERRNDS_A": [3, 4]},
    "food":      {"FDSCAT3_A": [2, 3]},
    "financial": {"PAYBLL12M_A": [1], "MEDDL12M_A": [1], "MEDNG12M_A": [1],
                  "RXDG12M_A": [1], "HOUSECOST_A": [1]},
    "transport": {"TRANSPOR_A": [1]},
}
label_cols = list(rules)

# Confirm that every variable we plan to use exists in the file
planned = set(label_sources + chronic_vars + ["PROXY_A", "ACCSSINT_A"])
print("Missing from file:", planned - set(df.columns) or "none, all present")

In [ ]:
# WHAT: Print the raw codes of each label variable, including blanks, and compare them with the codebook.
# HOW/WHY: value_counts(dropna=False) shows every code at once, so we can see which numbers mean "refused / not ascertained / don't know" (7, 8, 9) before using them.
for c in label_sources:
    print(c, df[c].value_counts(dropna=False).sort_index().to_dict())

## 4. Data cleaning (from raw file to modelling dataset)

Each step below is logged in a table at the end of this section, with the reason for it.

1. **Recode "refused / not ascertained / don't know" (7, 8, 9) to missing** in the label variables only. These numbers are placeholders, not answers, and a model would otherwise read them as real values. We do not recode the whole file, because 7 to 9 are genuine answers in other variables.
2. **Filter to chronic-condition patients**, the project's target population.
3. **Build the five labels** from the rules. A label stays *missing* if every source answer is missing, so an unknown is never counted as "no need".
4. **Drop patients with a missing label.** A model cannot learn from an unknown target, and guessing labels would invent ground truth.

In [ ]:
# WHAT: Turn the placeholder codes 7, 8 and 9 into missing values in the label variables, then keep only chronic-condition patients.
# HOW/WHY: The recode is limited to label_sources, where those codes never mean a real answer; the filter keeps rows where any condition variable equals 1 (Yes).
log = [("Raw NHIS 2023 Sample Adult file", len(df))]

df[label_sources] = df[label_sources].replace([7, 8, 9], np.nan)

d_all = df[(df[chronic_vars] == 1).any(axis=1)].copy()
log.append(("Kept adults with at least one chronic condition", len(d_all)))
print(log)

In [ ]:
# WHAT: Build the five 0/1 label columns from the rules, leaving a label missing when all its source answers are missing.
# HOW/WHY: isin() flags answers that signal a need and any() combines them (any signal = need); the NaN step stops "unknown" from silently becoming "no need".
def make_label(data, rule):
    hit = pd.concat([data[c].isin(v) for c, v in rule.items()], axis=1).any(axis=1)
    label = hit.astype(float)
    label[data[list(rule)].isna().all(axis=1)] = np.nan
    return label

for name, rule in rules.items():
    d_all[name] = make_label(d_all, rule)

print("Missing rate per label:")
print(d_all[label_cols].isna().mean().round(3))

# Drop patients with a missing label and record how many rows were lost
d = d_all.dropna(subset=label_cols).copy()
log.append(("Dropped adults with no usable answer for a label", len(d)))

cleaning_log = pd.DataFrame(log, columns=["Step", "Rows remaining"])
cleaning_log["Rows lost"] = (-cleaning_log["Rows remaining"].diff()).fillna(0).astype(int)
cleaning_log

**Label check.** A good label rule should agree with an independent measure. Patients with a *positive PHQ-2* (or GAD-2) screen should report depression (or anxiety) far more often than those with a negative screen. If that holds, code 1 really is the "positive" screen and the Emotional rule is sound.

In [ ]:
# WHAT: Compare the PHQ-2 and GAD-2 screeners with how often patients say they feel depressed or anxious.
# HOW/WHY: normalize="index" shows each screener group's answer mix as shares; the positive-screen row should have much higher shares in columns 1 and 2 (daily and weekly).
print(pd.crosstab(d["PHQ2SCREEN_A"], d["DEPFREQ_A"], normalize="index").round(2))
print()
print(pd.crosstab(d["GAD2SCREEN_A"], d["ANXFREQ_A"], normalize="index").round(2))

## 5. Exploratory data analysis (EDA)

Five observations that shape the modelling choices. The figures quoted come from our run on the full 2023 file and should reproduce when the notebook is run.

In [ ]:
# WHAT: Chart how common each need is among chronic-condition patients.
# HOW/WHY: The mean of a 0/1 column is the share of patients with that need; very uneven shares reveal class imbalance, which affects how we train and score.
rates = d[label_cols].mean().sort_values(ascending=False)
ax = (rates * 100).plot(kind="bar", color="#4C78A8", figsize=(7, 3.5), rot=0)
ax.set_ylabel("% of patients"); ax.set_title("Observation 1: how common is each need?")
for i, v in enumerate(rates * 100):
    ax.text(i, v + 0.5, f"{v:.1f}%", ha="center")
plt.tight_layout(); plt.show()

**Observation 1: class imbalance.** Emotional (about 32%) and financial (21%) needs are common, while food (9.4%), social (8.5%) and transport (7.3%) are rare. A model that predicts "no need" for the rare ones would still look accurate, so we use class weighting and judge by F1 and recall per need, not accuracy.

In [ ]:
# WHAT: Count how many of the five needs each patient has at the same time.
# HOW/WHY: summing the five 0/1 labels per row gives the count; if many patients have two or more, a multi-label approach is justified.
n_needs = d[label_cols].sum(axis=1).astype(int).value_counts().sort_index()
ax = n_needs.plot(kind="bar", color="#F58518", figsize=(6, 3.5), rot=0)
ax.set_xlabel("Number of needs"); ax.set_ylabel("Patients")
ax.set_title("Observation 2: needs per patient")
plt.tight_layout(); plt.show()
print("Share with 1+ needs:", round((d[label_cols].sum(axis=1) >= 1).mean(), 3))
print("Share with 2+ needs:", round((d[label_cols].sum(axis=1) >= 2).mean(), 3))

**Observation 2: patients often have several needs.** About 47% of patients have at least one need and about 20% have two or more. Predicting a single category would throw this away, which supports the **multi-label** design.

In [ ]:
# WHAT: Show how strongly each pair of needs tends to occur together.
# HOW/WHY: The correlation of two 0/1 columns is highest when the needs co-occur; this tells us whether the labels are independent enough for Binary Relevance.
corr = d[label_cols].corr()
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(corr, cmap="Blues", vmin=0, vmax=0.5)
ax.set_xticks(range(5)); ax.set_xticklabels(label_cols, rotation=45, ha="right")
ax.set_yticks(range(5)); ax.set_yticklabels(label_cols)
for i in range(5):
    for j in range(5):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")
ax.set_title("Observation 3: label correlations")
plt.tight_layout(); plt.show()

**Observation 3: needs cluster, but only moderately.** The strongest pairs are food with financial (0.36) and food with transport (0.27). These are moderate links, so treating the five needs as independent (Binary Relevance) is reasonable, and we test Classifier Chains later, which can use such links.

In [ ]:
# WHAT: Compare the patients we dropped (missing label) with those we kept, on age, sex and interview type.
# HOW/WHY: If dropped patients differ a lot, the cleaning could bias the model; PROXY_A shows whether someone else answered on the patient's behalf.
dropped = d_all.loc[~d_all.index.isin(d.index)]
comparison = pd.DataFrame({
    "Kept (n=%d)" % len(d): [d["AGEP_A"].mean(), (d["SEX_A"] == 2).mean(), (d["PROXY_A"] == 2).mean()],
    "Dropped (n=%d)" % len(dropped): [dropped["AGEP_A"].mean(), (dropped["SEX_A"] == 2).mean(), (dropped["PROXY_A"] == 2).mean()],
}, index=["Mean age", "Share coded sex=2", "Share coded proxy=2"]).round(3)
comparison

**Observation 4: the dropped patients are mostly a different interview type.** Age and sex are similar between kept and dropped patients, but about 32% of the dropped group have `PROXY_A` = 2 against 0.3% of the kept group. Those rows are not random: they are concentrated among a subgroup interviewed differently. The model therefore learns from patients who answered for themselves, which we note as a limitation. (Check the codebook for what code 2 of `PROXY_A` means.)

In [ ]:
# WHAT: Plot how each need varies with age group and with income band.
# HOW/WHY: pd.cut sorts patients into groups and groupby().mean() gives the share with each need per group; this checks that the patterns make sense before modelling.
age_groups = pd.cut(d["AGEP_A"], bins=[17, 39, 59, 74, 120], labels=["18-39", "40-59", "60-74", "75+"])
income_groups = pd.cut(d["RATCAT_A"], bins=[0, 4, 8, 14], labels=["codes 1-4", "codes 5-8", "codes 9-14"])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
(d.groupby(age_groups)[label_cols].mean() * 100).plot(kind="bar", ax=axes[0], rot=0, title="By age group")
(d.groupby(income_groups)[label_cols].mean() * 100).plot(kind="bar", ax=axes[1], rot=0, title="By income band (RATCAT_A)")
axes[0].set_ylabel("% with the need"); axes[1].legend(loc="upper right", fontsize=7)
fig.suptitle("Observation 5: needs by age and by income")
plt.tight_layout(); plt.show()

**Observation 5: income drives the economic needs.** Food, financial and transport needs fall steeply as the income band rises (lower `RATCAT_A` codes mean lower income relative to the poverty line; confirm in the codebook), which matches what we expect from the real world and later shows up as the top SHAP feature for food insecurity.

## 6. Preparing the input features

**Features** are what the model sees: age, sex, education, income ratio, region, urban/rural, general health, BMI category, smoking, marital status, household size, work, insurance, difficulty and disability, usual place of care, housing, pain, the 11 chronic-condition flags, and six mobility, sensory and internet variables added to help the Transportation need.

**Leakage rule.** Any variable used to build a label, or that nearly restates it (food-security items, medication-cost items, depression/anxiety medication and severity, PHQ item scores, delayed-care items, SNAP), is **excluded** from the features. Otherwise the model would simply rediscover our rule and the scores would be falsely high.

**Cleaning decisions for features.**
- Codes for "refused / unknown" become missing **column by column**, because 7 to 9 and 97 are real answers in some columns.
- Follow-up questions (asthma, cane/walker, wheelchair) are blank when the first question was "no", so a blank is turned into 0 rather than treated as missing.
- Home internet is blank for people with no internet at all, so those blanks become 0.
- Remaining small gaps (under 2%) are filled **after** the train/test split, using only training data.

In [ ]:
# WHAT: Recode the genuine "refused / unknown" codes to missing, column by column, and turn conditions into 0/1 flags.
# HOW/WHY: A dictionary of {column: missing codes} avoids damaging columns where 7-9 or 97 are valid answers; blank follow-up answers are set to 0 because the skip pattern means "no".
features = ["AGEP_A", "SEX_A", "EDUCP_A", "RATCAT_A", "REGION", "URBRRL", "PHSTAT_A",
            "BMICAT_A", "SMKCIGST_A", "MARSTAT_A", "PCNTADLT_A", "PCNTKIDS_A",
            "EMPLASTWK_A", "HICOV_A", "DIFF_A", "DISAB3_A", "USUALPL_A", "HOUTENURE_A",
            "PAIFRQ3M_A"] + chronic_vars
new_feats = ["EQUIP_A", "CANEWLKR_A", "WCHAIR_A", "VISIONDF_A", "HEARINGDF_A", "ACCSSHOM_A"]
features_v2 = features + new_feats

missing_map = {
    "SEX_A": [7, 9], "PHSTAT_A": [7, 9], "HICOV_A": [7, 9], "DIFF_A": [7, 9],
    "PAIFRQ3M_A": [7, 9], "EMPLASTWK_A": [7, 9], "EDUCP_A": [97, 99],
    "BMICAT_A": [9], "USUALPL_A": [9], "SMKCIGST_A": [9], "MARSTAT_A": [9],
    "HOUTENURE_A": [7, 8, 9],
}
for col, codes in missing_map.items():
    d[col] = d[col].replace(codes, np.nan)

d["ASTILL_A"] = d["ASTILL_A"].fillna(2)              # blank = never had asthma
for col in chronic_vars:
    d[col] = d[col].map({1: 1, 2: 0})                # 1 = has it, 0 = does not; 7/9 become NaN

In [ ]:
# WHAT: Recode the six extra mobility, sensory and internet features.
# HOW/WHY: Cane and wheelchair questions are only asked of equipment users, so blanks mean "no"; people with no internet at all (ACCSSINT_A = 2) were never asked the home-internet question, so they get 0.
d["EQUIP_A"] = d["EQUIP_A"].map({1: 1, 2: 0})
for c in ["CANEWLKR_A", "WCHAIR_A"]:
    d[c] = d[c].replace(9, np.nan)
    d.loc[d["EQUIP_A"] == 0, c] = 2
    d[c] = d[c].map({1: 1, 2: 0})
for c in ["VISIONDF_A", "HEARINGDF_A"]:
    d[c] = d[c].replace([7, 9], np.nan)
d["ACCSSHOM_A"] = d["ACCSSHOM_A"].replace(9, np.nan).map({1: 1, 2: 0})
d.loc[d["ACCSSINT_A"] == 2, "ACCSSHOM_A"] = 0

X = d[features_v2].copy()
y = d[label_cols].astype(int)

print("Features:", X.shape[1], "| Patients:", X.shape[0])
print("Missing values per feature (%), top 8:")
print((X.isna().mean() * 100).round(2).sort_values(ascending=False).head(8))

In [ ]:
# WHAT: Check whether any feature is almost perfectly tied to a label.
# HOW/WHY: A very high correlation with a label would signal leakage; the largest absolute correlation per feature should stay well below about 0.5.
corr_fl = pd.concat([X, y], axis=1).corr()[label_cols].drop(label_cols)
print(corr_fl.abs().max(axis=1).sort_values(ascending=False).head(8).round(2))

**Note on DISAB3_A and DIFF_A.** They correlate with the Social label (about 0.45) because that label is built from "difficulty with errands and social activities", which overlaps with disability. This is conceptual overlap rather than literal leakage. A robustness check in section 12 retrains without them.

## 7. Train/test split and preprocessing

We hold out **20% of patients for the final test**. The split is *stratified on the combination of needs* (for example "emotional + financial") so the training and test sets have the same mix. All filling of gaps, scaling and encoding is **fitted on training data only**, so no information leaks from the test set. The same preparation is reused for every model, which keeps the comparison fair.

In [ ]:
# WHAT: Split patients 80/20 into training and test sets, keeping the mix of need combinations the same in both.
# HOW/WHY: Stratifying on the label pattern (for example "01001") approximates multi-label stratification; patterns with fewer than 5 patients are grouped as "rare" because stratification needs at least two per group.
combo = y.astype(str).agg("".join, axis=1)
combo = combo.where(combo.map(combo.value_counts()) >= 5, "rare")
if (combo == "rare").sum() < 2:                      # a lone "rare" patient joins the most common pattern
    combo = combo.replace("rare", combo.value_counts().index[0])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=combo)

print(X_train.shape, X_test.shape)
print(pd.DataFrame({"train": y_train.mean(), "test": y_test.mean()}).round(3))

In [ ]:
# WHAT: Define how each kind of feature is prepared: numbers are gap-filled with the median and scaled, categories are gap-filled with the most common value and one-hot encoded.
# HOW/WHY: Unordered codes (region, marital status, and so on) must not be treated as numbers, so they are one-hot encoded; fitting this inside each model's pipeline keeps cross-validation honest.
nominal = ["SEX_A", "REGION", "MARSTAT_A", "SMKCIGST_A", "HOUTENURE_A",
           "USUALPL_A", "EMPLASTWK_A", "HICOV_A"]
numeric = [f for f in features_v2 if f not in nominal]

preprocess = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), numeric),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), nominal),
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)

def build(estimator):
    # One copy of the estimator per label = Binary Relevance
    return Pipeline([("prep", preprocess), ("clf", MultiOutputClassifier(estimator))])

def metric_row(name, pred):
    return {"Model": name,
            "Test Macro F1": f1_score(y_test, pred, average="macro", zero_division=0),
            "Test Micro F1": f1_score(y_test, pred, average="micro", zero_division=0),
            "Precision": precision_score(y_test, pred, average="macro", zero_division=0),
            "Recall": recall_score(y_test, pred, average="macro", zero_division=0),
            "Hamming Loss": hamming_loss(y_test, pred),
            "Subset Acc": accuracy_score(y_test, pred)}

## 8. Base model: Random Forest with Binary Relevance

Five forests (one per need), 200 trees each, with `class_weight="balanced"` so the rare needs are not ignored and `min_samples_leaf=5` to stop single trees memorising tiny groups.

**Validation.** 5-fold cross-validation on the training set. A model that does well in every fold is not just lucky. The test set is touched only afterwards.

In [ ]:
# WHAT: Build the base Random Forest and score it with 5-fold cross-validation on the training data only.
# HOW/WHY: Random Forest averages many trees, which captures non-linear links in survey data and resists overfitting; cross-validation checks it on five different splits before we look at the test set.
rf = build(RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                                  class_weight="balanced", n_jobs=-1, random_state=42))

scores = cross_validate(rf, X_train, y_train, cv=cv,
                        scoring=["f1_macro", "f1_micro", "precision_macro", "recall_macro"])
for k in ["f1_macro", "f1_micro", "precision_macro", "recall_macro"]:
    s = scores["test_" + k]
    print(f"{k}: {s.mean():.3f} (+/- {s.std():.3f})")

In [ ]:
# WHAT: Fit the base model on all training data and report per-need results on the held-out test set.
# HOW/WHY: The per-need report shows which needs the model finds well and which it misses, which overall scores can hide.
rf.fit(X_train, y_train)
print(classification_report(y_test, rf.predict(X_test), target_names=label_cols, zero_division=0))

## 9. Comparing other models (same process)

To check that Random Forest is a sensible choice, every model below uses the **same features, preprocessing, split and cross-validation**.

| Model | Why it is included |
|---|---|
| Predict none | Floor: what a model that never finds a need scores |
| Decision Tree | Simple, easy-to-read baseline |
| Logistic Regression | Tests whether a linear model is enough |
| Random Forest | Our base model |
| RF native multi-output | One forest trained on all five needs together |
| RF + Classifier Chain | Each need's prediction feeds the next, to use the links between needs |

In [ ]:
# WHAT: Train and score six models on identical data, recording cross-validated Macro F1, test metrics, and per-need F1.
# HOW/WHY: Decision Tree is the simple baseline, Logistic Regression the linear check, and the two forest variants test other multi-label strategies; the dummy model sets the floor.
models = {
    "Predict none (baseline)": build(DummyClassifier(strategy="most_frequent")),
    "Decision Tree": build(DecisionTreeClassifier(max_depth=6, min_samples_leaf=20,
                                                  class_weight="balanced", random_state=42)),
    "Logistic Regression": build(LogisticRegression(max_iter=1000, class_weight="balanced")),
    "Random Forest (default)": rf,
    "RF native multi-output": Pipeline([("prep", preprocess), ("clf", RandomForestClassifier(
        n_estimators=200, min_samples_leaf=5, class_weight="balanced", n_jobs=-1, random_state=42))]),
    "RF + Classifier Chain": Pipeline([("prep", preprocess), ("clf", ClassifierChain(
        RandomForestClassifier(n_estimators=200, min_samples_leaf=5, class_weight="balanced",
                               n_jobs=-1, random_state=42), order="random", random_state=42))]),
}

rows, per_label = [], {}
for name, m in models.items():
    cvs = cross_validate(m, X_train, y_train, cv=cv, scoring=["f1_macro"])
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    rows.append({**metric_row(name, pred), "CV Macro F1": cvs["test_f1_macro"].mean()})
    per_label[name] = f1_score(y_test, pred, average=None, zero_division=0)

comparison_table = pd.DataFrame(rows).set_index("Model")
comparison_table = comparison_table[["CV Macro F1", "Test Macro F1", "Test Micro F1", "Precision",
                                     "Recall", "Hamming Loss", "Subset Acc"]].round(3)
comparison_table

In [ ]:
# WHAT: Show each model's F1 score for each of the five needs in one table and one grouped bar chart.
# HOW/WHY: Overall scores can hide which need a model handles badly; the chart makes the differences between models and needs easy to compare.
per_label_df = pd.DataFrame(per_label, index=label_cols).drop(columns="Predict none (baseline)").round(3)
display(per_label_df)
per_label_df.plot(kind="bar", figsize=(10, 4), rot=0, title="F1 by need and model (test set)")
plt.ylabel("F1"); plt.legend(fontsize=7); plt.tight_layout(); plt.show()

**Reading the comparison.**
- Random Forest beats the Decision Tree and Logistic Regression on Macro F1, and by a wide margin on precision (the simpler models flag many people who do not need help).
- The two forest variants score about the same, so the choice of multi-label strategy is not what limits performance. We keep **Binary Relevance**, which gives one probability per need and fits the app (each suggestion can be accepted or declined on its own).
- The "predict none" baseline has a *better* Hamming loss and subset accuracy than every real model. This is expected: most patients have no need, so "no need" is often right. It scores 0 on F1 because it never finds anyone. That is why we use **Macro F1 and per-need recall** as the main measures.
- Transport stays weak for every model, which suggests the survey features carry little information about transport barriers.

## 10. Tuning the base model

We try 15 random combinations of Random Forest settings (number of trees, tree depth, minimum leaf size, features per split) with 3-fold cross-validation on the training set, scoring on **Macro F1**. Random search covers a wide range quickly, where trying every combination would take far longer.

In [ ]:
# WHAT: Search over Random Forest settings using 3-fold cross-validation on the training set only.
# HOW/WHY: Random search tries 15 sampled combinations instead of every one, which is much faster and usually finds a good setting; scoring on Macro F1 keeps the rare needs in focus.
param_dist = {
    "clf__estimator__n_estimators": randint(150, 400),
    "clf__estimator__max_depth": [8, 12, 16, None],
    "clf__estimator__min_samples_leaf": randint(3, 25),
    "clf__estimator__max_features": ["sqrt", 0.3, 0.5],
}
search = RandomizedSearchCV(rf, param_dist, n_iter=15, scoring="f1_macro",
                            cv=3, random_state=42, n_jobs=1, verbose=1)
search.fit(X_train, y_train)

best_rf = search.best_estimator_
print("Best CV Macro F1:", round(search.best_score_, 3))
print(search.best_params_)
print("Tuned test Macro F1:", round(f1_score(y_test, best_rf.predict(X_test), average="macro", zero_division=0), 3))

## 11. Choosing a cut-off for each need

The forest outputs a probability. "Flag this need if the probability is at least 0.5" is rarely best for a rare need. We choose the cut-off **per need** by maximising F1 on *out-of-fold* predictions (each patient scored by a model that never saw them), using **training data only**. The cut-offs are then frozen and applied to the test set once.

In [ ]:
# WHAT: Collect out-of-fold probabilities for the tuned model, then pick the F1-maximising cut-off for each need.
# HOW/WHY: Out-of-fold scores mean the cut-offs are not tuned on memorised data; a simple grid scan from 0.10 to 0.90 is transparent and easy to explain.
oof = np.zeros(y_train.shape)
for tr, va in cv.split(X_train):
    m = clone(best_rf).fit(X_train.iloc[tr], y_train.iloc[tr])
    oof[va] = np.column_stack([p[:, 1] for p in m.predict_proba(X_train.iloc[va])])

grid = np.arange(0.1, 0.91, 0.02)
thresholds = {}
for j, lab in enumerate(label_cols):
    f1s = [f1_score(y_train.iloc[:, j], oof[:, j] >= t, zero_division=0) for t in grid]
    thresholds[lab] = round(float(grid[int(np.argmax(f1s))]), 2)
print("Cut-offs:", thresholds)

# Apply the frozen cut-offs to the test set
test_probs = np.column_stack([p[:, 1] for p in best_rf.predict_proba(X_test)])
final_pred = np.column_stack([test_probs[:, j] >= thresholds[l]
                              for j, l in enumerate(label_cols)]).astype(int)

## 12. Final results and robustness

The final engine is the **tuned Random Forest (Binary Relevance) with per-need cut-offs**. Below it is placed in the comparison table with the other models.

In [ ]:
# WHAT: Add the tuned model, and the tuned model with cut-offs, to the comparison table and chart the headline scores.
# HOW/WHY: Scoring everything on the same test patients makes the table a fair end-to-end comparison; a bar chart of Macro and Micro F1 gives the at-a-glance view of how the models rank.
tuned_default = best_rf.predict(X_test)
extra = pd.DataFrame([metric_row("Random Forest (tuned)", tuned_default),
                      metric_row("Random Forest (tuned + cut-offs)", final_pred)]).set_index("Model")
final_table = pd.concat([comparison_table.drop(columns="CV Macro F1"), extra.round(3)])
display(final_table)

per_label_final = per_label_df.copy()
per_label_final["RF (tuned)"] = f1_score(y_test, tuned_default, average=None, zero_division=0).round(3)
per_label_final["RF (tuned + cut-offs)"] = f1_score(y_test, final_pred, average=None, zero_division=0).round(3)
display(per_label_final)

ax = final_table[["Test Macro F1", "Test Micro F1"]].plot(kind="barh", figsize=(8, 4.5))
ax.set_title("Model comparison on the held-out test set"); ax.set_xlabel("F1")
plt.tight_layout(); plt.show()

In [ ]:
# WHAT: Print the final engine's per-need precision, recall and F1, plus Hamming loss and subset accuracy.
# HOW/WHY: This is the model that goes into the app, so we report it in full; recall above precision is intentional, because missing a patient who needs help costs more than a false alarm.
print(classification_report(y_test, final_pred, target_names=label_cols, zero_division=0))
print("Hamming loss:", round(hamming_loss(y_test, final_pred), 3),
      "| Subset accuracy:", round(accuracy_score(y_test, final_pred), 3))

**Robustness check: does the Social need rely on overlapping features?** We retrain without the disability and mobility variables (`DISAB3_A`, `DIFF_A`, `EQUIP_A`, `CANEWLKR_A`, `WCHAIR_A`). If Social F1 collapses, the model mostly relearns the label's definition. If it holds up, other features carry real signal.

In [ ]:
# WHAT: Retrain the tuned model without the five disability and mobility features and compare per-need F1.
# HOW/WHY: set_params swaps in a preprocessing step that excludes those columns while keeping the tuned forest settings, so only the features change.
overlap = ["DISAB3_A", "DIFF_A", "EQUIP_A", "CANEWLKR_A", "WCHAIR_A"]
keep = [f for f in features_v2 if f not in overlap]
prep_no_overlap = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]),
     [f for f in keep if f not in nominal]),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), nominal)])

m3 = clone(best_rf).set_params(prep=prep_no_overlap).fit(X_train[keep], y_train)
p3 = m3.predict(X_test[keep])
print("Without overlap features, test Macro F1:", round(f1_score(y_test, p3, average="macro", zero_division=0), 3))
print(dict(zip(label_cols, f1_score(y_test, p3, average=None, zero_division=0).round(3))))
print("With them (default cut-off):", round(f1_score(y_test, tuned_default, average="macro", zero_division=0), 3))
print(dict(zip(label_cols, f1_score(y_test, tuned_default, average=None, zero_division=0).round(3))))

## 13. Explaining the predictions with SHAP

**What SHAP does.** For one patient, SHAP gives each answer a score showing how much it pushed that need's probability up or down, like splitting a restaurant bill fairly among everyone who ordered. Averaging these scores over many patients shows which answers the model relies on most for each need.

**How to read the plots.** Features are ranked top to bottom by importance. Each dot is a patient: red = high value of the feature, blue = low. Dots to the right of zero push the need *up*.

**Caution.** SHAP shows what the *model relies on*, not what *causes* a need. We describe features as "associated with" predictions.

In [ ]:
# WHAT: Compute SHAP values for each of the five forests on a sample of 500 test patients and draw one summary plot per need.
# HOW/WHY: TreeExplainer is the exact, fast method for tree models; Binary Relevance gives one forest per need, so each need gets its own explanation.
prep_fit = best_rf.named_steps["prep"]
Xt = prep_fit.transform(X_test)
Xt = Xt.toarray() if hasattr(Xt, "toarray") else np.asarray(Xt)
names = [n.split("__", 1)[1] for n in prep_fit.get_feature_names_out()]
Xs = Xt[:500]

top5 = {}
for j, lab in enumerate(label_cols):
    est = best_rf.named_steps["clf"].estimators_[j]
    sv = shap.TreeExplainer(est).shap_values(Xs)
    sv = sv[1] if isinstance(sv, list) else (sv[:, :, 1] if sv.ndim == 3 else sv)
    shap.summary_plot(sv, Xs, feature_names=names, show=False, max_display=10)
    plt.title(f"SHAP summary: {lab}"); plt.tight_layout(); plt.show()
    top5[lab] = pd.Series(np.abs(sv).mean(axis=0), index=names).sort_values(ascending=False).head(5).round(3).to_dict()

for lab, v in top5.items():
    print(lab, "->", v)

**What the explanations show (our run).**
- **Food:** income level is by far the strongest factor, followed by age and housing situation.
- **Financial:** age, income level, pain frequency and general health.
- **Emotional:** age, pain frequency, general health, and vision difficulty.
- **Social:** disability, general health, walking difficulty and mobility aids, which reflects how the label is built (see the robustness check).
- **Transport:** income level, general health, pain and housing. The mobility variables we added did not rank, which matches the weak Transport score.

## 14. Saving the engine for the app

The mobile app does not run this notebook. It calls a small web service (FastAPI) that loads three saved files: the trained pipeline (preprocessing plus the five forests), the cut-offs, and the list of input features and need names. **Use the same scikit-learn version when loading them**.

In [ ]:
# WHAT: Save the fitted pipeline, the per-need cut-offs, and the feature and label lists to files.
# HOW/WHY: joblib stores the whole fitted pipeline in one file so the backend can score new patients with exactly the same preparation; the cut-offs and lists are plain JSON so they can be inspected.
joblib.dump(best_rf, "support_needs_model.joblib")
json.dump(thresholds, open("thresholds.json", "w"))
json.dump({"features": features_v2, "labels": label_cols}, open("model_meta.json", "w"))
print("Saved. scikit-learn version used for training:", sklearn.__version__)

## 15. Using the engine like the app does

The cells below **load the saved files** (not the training variables) and score patients described in plain language, just as the app sends them. This is the same logic the FastAPI backend runs.

**How a result is produced (30-second explanation).**
1. The patient's answers are turned into the survey codes the model was trained on, and any blank answer is filled using the training data.
2. Five Random Forest models, one per support need, each estimate the probability that the patient has that need.
3. A need is flagged when its probability passes the cut-off tuned for that need.
4. For each flagged need, SHAP lists the answers that pushed its probability up the most.

In [ ]:
# WHAT: Load the saved engine from disk and prepare one SHAP explainer per need.
# HOW/WHY: Loading from the files (not reusing the training variables) proves the saved files are enough to run the engine on their own, which is what the backend depends on.
model = joblib.load("support_needs_model.joblib")
cutoffs = json.load(open("thresholds.json"))
meta = json.load(open("model_meta.json"))
FEATURES, LABELS = meta["features"], meta["labels"]
prep_loaded = model.named_steps["prep"]
explainers = [shap.TreeExplainer(e) for e in model.named_steps["clf"].estimators_]
feat_names = [n.split("__", 1)[1] for n in prep_loaded.get_feature_names_out()]

CATEGORY = {"emotional": "Emotional & mental wellbeing", "social": "Social connection",
            "food": "Food & nutrition", "financial": "Financial assistance",
            "transport": "Transportation & healthcare access"}
FRIENDLY = {"AGEP_A": "Age", "RATCAT_A": "Income level", "PHSTAT_A": "General health",
            "PAIFRQ3M_A": "Pain frequency", "DISAB3_A": "Disability", "DIFF_A": "Walking difficulty",
            "VISIONDF_A": "Vision difficulty", "HOUTENURE_A": "Housing situation", "EDUCP_A": "Education",
            "EQUIP_A": "Mobility aid use", "CANEWLKR_A": "Mobility aid use", "ACCSSHOM_A": "Home internet",
            "HYPEV_A": "Hypertension", "DIBEV_A": "Diabetes", "CHDEV_A": "Heart disease",
            "ARTHEV_A": "Arthritis", "COPDEV_A": "COPD", "ASTILL_A": "Asthma",
            "CANEV_A": "Cancer", "STREV_A": "Stroke", "CHLEV_A": "High cholesterol",
            "ANGEV_A": "Angina", "MIEV_A": "Heart attack", "HEARINGDF_A": "Hearing difficulty",
            "SMKCIGST_A": "Smoking status", "MARSTAT_A": "Marital status", "BMICAT_A": "Weight category",
            "SEX_A": "Sex", "REGION": "Region", "URBRRL": "Urban/rural area", "HICOV_A": "Health insurance",
            "EMPLASTWK_A": "Work status", "USUALPL_A": "Usual place of care",
            "PCNTADLT_A": "Adults in household", "PCNTKIDS_A": "Children in household"}

In [ ]:
# WHAT: Translate a patient's plain-language answers into the survey codes the model expects.
# HOW/WHY: Lookup dictionaries keep the translation auditable against the codebook; anything the patient does not answer stays blank and the saved preprocessing fills it in.
SEX = {"male": 1, "female": 2}
INCOME = {"very_low": 2, "low": 5, "middle": 8, "high": 12}
HEALTH = {"excellent": 1, "very_good": 2, "good": 3, "fair": 4, "poor": 5}
PAIN = {"never": 1, "some_days": 2, "most_days": 3, "every_day": 4}
DIFF = {"none": 1, "some": 2, "a_lot": 3, "cannot": 4}
HOUSING = {"own": 1, "rent": 2, "other": 3}
COND = {"hypertension": "HYPEV_A", "diabetes": "DIBEV_A", "heart_disease": "CHDEV_A",
        "asthma": "ASTILL_A", "copd": "COPDEV_A", "arthritis": "ARTHEV_A",
        "cancer": "CANEV_A", "stroke": "STREV_A", "high_cholesterol": "CHLEV_A"}

def to_row(a):
    r = {f: np.nan for f in FEATURES}
    walk, vis = a.get("walking_difficulty", "none"), a.get("vision_difficulty", "none")
    aid = int(a.get("uses_mobility_aid", False))
    r.update({"AGEP_A": a["age"], "SEX_A": SEX[a["sex"]], "RATCAT_A": INCOME[a["income_level"]],
              "PHSTAT_A": HEALTH[a["general_health"]], "PAIFRQ3M_A": PAIN[a.get("pain", "never")],
              "DIFF_A": DIFF[walk], "VISIONDF_A": DIFF[vis],
              "DISAB3_A": 1 if max(DIFF[walk], DIFF[vis]) >= 3 else 2,
              "HOUTENURE_A": HOUSING[a.get("housing", "own")],
              "ACCSSHOM_A": int(a.get("home_internet", True)),
              "EQUIP_A": aid, "CANEWLKR_A": aid, "WCHAIR_A": 0,
              "HICOV_A": 1 if a.get("has_insurance", True) else 2,
              "EMPLASTWK_A": 1 if a.get("employed", False) else 2})
    for v in chronic_vars:
        r[v] = 0
    for c in a.get("conditions", []):
        r[COND[c]] = 1
    return pd.DataFrame([r])[FEATURES]

In [ ]:
# WHAT: Score one patient: return each need's probability, whether it is flagged, and the top answers behind each flagged need.
# HOW/WHY: Flagging uses each need's own cut-off; SHAP values for the flagged need's forest are ranked and the top three positive contributors become the "main drivers".
def predict_patient(answers):
    row = to_row(answers)
    probs = [float(p[0, 1]) for p in model.predict_proba(row)]
    xt = prep_loaded.transform(row)
    xt = xt.toarray() if hasattr(xt, "toarray") else np.asarray(xt)
    out = []
    for j, lab in enumerate(LABELS):
        flagged = probs[j] >= cutoffs[lab]
        drivers = []
        if flagged:
            sv = explainers[j].shap_values(xt)
            sv = sv[1][0] if isinstance(sv, list) else (sv[0, :, 1] if sv.ndim == 3 else sv[0])
            for n in pd.Series(sv, index=feat_names).sort_values(ascending=False).head(3).index:
                name = FRIENDLY.get(re.sub(r"_\d+(\.\d+)?$", "", n), n)
                if name not in drivers:
                    drivers.append(name)
        out.append({"Support area": CATEGORY[lab], "Probability": round(probs[j], 2),
                    "Cut-off": cutoffs[lab], "Flagged": flagged, "Main drivers": ", ".join(drivers)})
    return pd.DataFrame(out)

In [ ]:
# WHAT: Run three contrasting patients through the engine: low need, high need, and in between.
# HOW/WHY: Contrasting patients show that the output changes sensibly with the answers; the middle patient shows the engine does not simply flag everything.
patients = {
    "Patient A: 30, high income, excellent health": {
        "age": 30, "sex": "male", "income_level": "high", "general_health": "excellent",
        "conditions": ["asthma"]},
    "Patient B: 67, very low income, several conditions": {
        "age": 67, "sex": "female", "income_level": "very_low", "general_health": "fair",
        "pain": "most_days", "walking_difficulty": "a_lot", "housing": "rent",
        "home_internet": False, "uses_mobility_aid": True,
        "conditions": ["diabetes", "hypertension", "arthritis"]},
    "Patient C: 55, middle income, one condition": {
        "age": 55, "sex": "female", "income_level": "middle", "general_health": "fair",
        "pain": "some_days", "conditions": ["hypertension"]},
}
for title, answers in patients.items():
    print(title)
    display(predict_patient(answers))

## 16. Limitations

1. **Social is a proxy.** The 2023 file has no loneliness or social-support question, so the Social label measures difficulty with social participation and daily tasks, and the model leans heavily on disability variables.
2. **Dropped patients are not random.** About 4.4% of chronic-condition patients were dropped for missing labels, mostly proxy interviews, so the model learns from self-responding patients.
3. **Transport has the weakest signal.** No variable captures distance to services, car ownership or local transit.
4. **Labels are self-reported survey answers**, and a "need" is a possibility to review, not a confirmed circumstance.
5. **US data.** NHIS represents US adults, so predictions need validation before use with Kenyan patients.
6. **Small differences.** Gaps of around 0.01 in Macro F1 (for example tuned against default settings) are within the spread of the cross-validation scores, so only the larger gaps (forests against the simpler models) should be treated as reliable.
7. **Cut-offs and tuning** were chosen on training data only, but the final scores come from a single test split.

## 17. Conclusion

A tuned Random Forest with one model per need (Binary Relevance) and need-specific cut-offs gave the best Macro F1 of the models compared. It identifies emotional and financial needs best and transport needs least well. SHAP shows the model relies on plausible factors (income for food and financial needs, health and function for emotional and social needs). The saved engine is served through a FastAPI backend and used by the Android app, where patients can review, correct, or decline each suggested support area.

*Architecture:* Android app (Kotlin) → FastAPI `/predict` → saved pipeline + cut-offs → probabilities, flags and main drivers → result cards in the app.